# STEP 2B: EfficientNet-B0 Retinal Cropping at 384×384 Input Resolution

This notebook executes **STEP 2B**: Controlled experiment evaluating the impact of:
1. **Retinal Cropping**: Automated elimination of 10%–38% empty black camera margins with aspect-ratio preserving square padding.
2. **Higher Resolution (384×384)**: 2.94× more spatial pixel density than 224×224, significantly enhancing the visibility of subtle microaneurysms and exudates.
3. **Two-Stage Fine-Tuning**: Reusing the validated Stage 1 Head Warmup + Stage 2 Full Fine-Tuning protocol.

### Experimental Controls:
- Model: EfficientNet-B0 (ImageNet-pretrained weights)
- Batch size: 16 (optimized for 384×384 within GPU VRAM)
- Mixed Precision: Enabled where supported (`fp16`)
- Splits: Identical `train.csv` (2,563) and `val.csv` (549). Test set `test.csv` is strictly held out.
- Checkpoints: `efficientnet_b0_cropped384_stage1_best.pth` & `stage2_best.pth` (existing baselines preserved).

## 1. Environment & GPU Configuration

In [1]:
import os
import sys

# ---- Mount Google Drive (if in Colab) ----
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("Google Drive mounted successfully.")
except ImportError:
    print("Running outside Google Colab.")

# ---- Project Root Configuration ----
PROJECT_ROOT = "/content/drive/MyDrive/DR_project"
if not os.path.isdir(PROJECT_ROOT):
    for cand in ["/content/drive/MyDrive/DR_Project", "/content/drive/Shareddrives/DR_project", "."]:
        if os.path.isdir(cand):
            PROJECT_ROOT = os.path.abspath(cand)
            break

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import torch
import torchvision

print("=" * 65)
print("              HARDWARE & ENVIRONMENT CONFIGURATION             ")
print("=" * 65)
print(f"  Project Root:        {PROJECT_ROOT}")
print(f"  Python Version:      {sys.version.split()[0]}")
print(f"  PyTorch Version:     {torch.__version__}")
print(f"  CUDA Available:      {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  GPU Device:          {torch.cuda.get_device_name(0)}")
    print(f"  GPU Memory:          {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
    device = torch.device('cuda')
else:
    print("  GPU Device:          None (CPU fallback)")
    device = torch.device('cpu')
print("=" * 65)

Mounted at /content/drive
Google Drive mounted successfully.
              HARDWARE & ENVIRONMENT CONFIGURATION             
  Project Root:        /content/drive/MyDrive/DR_project
  Python Version:      3.13.15
  PyTorch Version:     2.11.0+cu130
  CUDA Available:      True
  GPU Device:          Tesla T4
  GPU Memory:          14.56 GB


## 2. Imports, Output Directories, and Reproducibility Seed

In [2]:
import gc
import json
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.training.trainer import set_seed
from src.data.dataloader_v2 import get_crop_dataloaders
from src.models.efficientnet import build_efficientnet_b0
from src.training.two_stage_pipeline import run_efficientnet_two_stage_pipeline
from src.evaluation.evaluator import evaluate_model
from src.evaluation.metrics import DR_CLASS_NAMES

SEED = 42
set_seed(SEED)

CKPT_DIR = os.path.join(PROJECT_ROOT, "checkpoints")
METRICS_DIR = os.path.join(PROJECT_ROOT, "results", "metrics")
PLOTS_DIR = os.path.join(PROJECT_ROOT, "results", "plots")

for d in [CKPT_DIR, METRICS_DIR, PLOTS_DIR]:
    os.makedirs(d, exist_ok=True)

print(f"Output directories verified.")
print(f"Global random seed set to: {SEED}")

Output directories verified.
Global random seed set to: 42


## 3. DataLoaders Setup (Cropped 384×384 Resolution)

Initializes training and validation DataLoaders with **retinal cropping enabled**, **square-padded aspect ratio**, and **384×384 bicubic interpolation**. Mini-batch size is set to 16.

In [3]:
SPLITS_DIR = os.path.join(PROJECT_ROOT, "src", "data", "splits")
RAW_IMG_DIR = os.path.join(PROJECT_ROOT, "src", "data", "raw", "train_images")

BATCH_SIZE = 16
IMG_SIZE = 384
NUM_WORKERS = 2 if os.name != 'nt' else 0

dataloaders = get_crop_dataloaders(
    splits_dir=SPLITS_DIR,
    img_dir=RAW_IMG_DIR,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    img_size=IMG_SIZE,
    do_crop=True,
    preserve_aspect_ratio=True,
    interpolation="bicubic",
    pin_memory=torch.cuda.is_available(),
)

train_loader = dataloaders["train"]
val_loader = dataloaders["val"]

print("=" * 65)
print("             CROPPED 384×384 DATASET SIZES & BATCHES           ")
print("=" * 65)
print(f"  Train Samples:      {len(train_loader.dataset):>5} ({len(train_loader)} batches of {BATCH_SIZE})")
print(f"  Validation Samples: {len(val_loader.dataset):>5} ({len(val_loader)} batches of {BATCH_SIZE})")
print(f"  Input Resolution:   {IMG_SIZE} × {IMG_SIZE}")
print(f"  Retinal Cropping:   Enabled (Bicubic + Aspect Preserved)")
print("=" * 65)

             CROPPED 384×384 DATASET SIZES & BATCHES           
  Train Samples:       2563 (161 batches of 16)
  Validation Samples:   549 (35 batches of 16)
  Input Resolution:   384 × 384
  Retinal Cropping:   Enabled (Bicubic + Aspect Preserved)


## 4. Execute Two-Stage Fine-Tuning Pipeline (Cropped 384×384)

- **Stage 1**: 5 epochs head warmup (`lr = 1e-3`, frozen backbone, frozen BN stats, unweighted CrossEntropyLoss)
- **Stage 2**: 30 epochs full fine-tuning (`backbone_lr = 1e-5`, `head_lr = 1e-4`, cosine warmup 3 epochs, patience = 7)
- Saved checkpoints:
  - `checkpoints/efficientnet_b0_cropped384_stage1_best.pth`
  - `checkpoints/efficientnet_b0_cropped384_stage2_best.pth`

In [ ]:
start_time = time.time()

results = run_efficientnet_two_stage_pipeline(
    train_loader=train_loader,
    val_loader=val_loader,
    checkpoint_dir=CKPT_DIR,
    stage1_filename="efficientnet_b0_cropped384_stage1_best.pth",
    stage2_filename="efficientnet_b0_cropped384_stage2_best.pth",
    stage1_epochs=5,
    stage2_epochs=30,
    stage2_warmup_epochs=3,
    stage1_lr=1e-3,
    stage2_backbone_lr=1e-5,
    stage2_head_lr=1e-4,
    weight_decay=1e-4,
    early_stopping_patience=7,
    device=device,
    seed=SEED,
)

total_duration = time.time() - start_time
print(f"\nCropped 384x384 training completed in {total_duration:.1f}s ({total_duration/60:.2f} min).")

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 146MB/s]



      STAGE 1: CLASSIFICATION HEAD WARMUP (FROZEN BACKBONE)      
  Total Parameters:      4,013,953
  Trainable Parameters:  6,405 (Head only)
  Frozen Parameters:     4,007,548 (Backbone)
            TRAINING STARTED [STAGE1_WARMUP]            
  Device:              cuda
  Mixed Precision:     True
  Max Epochs:          5
  Early Stop Patience: 0
  Checkpoint File:     efficientnet_b0_cropped384_stage1_best.pth
  Freeze BatchNorm:    True
  LR (Group 0):          1.00e-03
  [stage1_warmup] Epoch [ 1/ 5]  Train Loss: 0.8098  Train Acc: 0.7054  |  Val Loss: 0.7097  Val Acc: 0.7231  |  LR: 0.001000  (2153.8s)
         -> New best stage1_warmup model saved! (val_loss improved by inf) -> /content/drive/MyDrive/DR_project/checkpoints/efficientnet_b0_cropped384_stage1_best.pth
  [stage1_warmup] Epoch [ 2/ 5]  Train Loss: 0.6329  Train Acc: 0.7675  |  Val Loss: 0.7037  Val Acc: 0.7304  |  LR: 0.001000  (880.1s)
         -> New best stage1_warmup model saved! (val_loss improved by 0.0060) 

## 5. Save Histories and Generate Training Curves

In [ ]:
# 1. Save Histories to CSV
h1_path = os.path.join(METRICS_DIR, "efficientnet_b0_cropped384_stage1_history.csv")
h2_path = os.path.join(METRICS_DIR, "efficientnet_b0_cropped384_stage2_history.csv")

df_h1 = pd.DataFrame(results["history_stage1"])
df_h1.insert(0, "epoch", range(1, len(df_h1) + 1))
df_h1.to_csv(h1_path, index=False)

df_h2 = pd.DataFrame(results["history_stage2"])
df_h2.insert(0, "epoch", range(1, len(df_h2) + 1))
df_h2.to_csv(h2_path, index=False)
print(f"Saved Stage 1 history: {h1_path}")
print(f"Saved Stage 2 history: {h2_path}")

# 2. Combined Two-Stage Training Curves
h1 = results["history_stage1"]
h2 = results["history_stage2"]
s1_len = len(h1["train_loss"])
s2_len = len(h2["train_loss"])
x = list(range(1, s1_len + s2_len + 1))

train_loss = list(h1["train_loss"]) + list(h2["train_loss"])
val_loss = list(h1["val_loss"]) + list(h2["val_loss"])
train_acc = list(h1["train_accuracy"]) + list(h2["train_accuracy"])
val_acc = list(h1["val_accuracy"]) + list(h2["val_accuracy"])

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8), sharex=True)

ax1.plot(x, train_loss, 'o-', color='#1f77b4', label='Train Loss', linewidth=2)
ax1.plot(x, val_loss, 's-', color='#ff7f0e', label='Validation Loss', linewidth=2)
ax1.axvline(x=s1_len + 0.5, color='gray', linestyle='--', linewidth=1.5, label='Stage Transition')
ax1.set_ylabel('Loss', fontsize=12)
ax1.set_title('EfficientNet-B0 (Cropped 384×384): Two-Stage Training Progression', fontsize=14, fontweight='bold')
ax1.grid(True, linestyle='--', alpha=0.5)
ax1.legend(loc='upper right', fontsize=10)

ax2.plot(x, train_acc, 'o-', color='#2ca02c', label='Train Accuracy', linewidth=2)
ax2.plot(x, val_acc, 's-', color='#d62728', label='Validation Accuracy', linewidth=2)
ax2.axvline(x=s1_len + 0.5, color='gray', linestyle='--', linewidth=1.5, label='Stage Transition')
ax2.set_xlabel('Epoch (Stage 1 Warmup -> Stage 2 Full Fine-Tuning)', fontsize=12)
ax2.set_ylabel('Accuracy', fontsize=12)
ax2.grid(True, linestyle='--', alpha=0.5)
ax2.legend(loc='lower right', fontsize=10)

plt.tight_layout()
combined_plot_path = os.path.join(PLOTS_DIR, "efficientnet_b0_cropped384_combined_curves.png")
plt.savefig(combined_plot_path, dpi=300)
plt.show()

## 6. Validation Evaluation on Best Stage 2 Model (Cropped 384×384)

Evaluates the best Stage 2 checkpoint strictly on `val.csv` (549 samples) to compute Accuracy, Macro Precision, Recall, Macro F1, QWK, per-class sensitivity & specificity, and confusion matrix.

In [ ]:
best_stage2_model = build_efficientnet_b0(num_classes=5, pretrained=False)
ckpt = torch.load(results["stage2_checkpoint_path"], map_location=device, weights_only=False)
best_stage2_model.load_state_dict(ckpt["model_state_dict"])

val_eval = evaluate_model(
    model=best_stage2_model,
    dataloader=val_loader,
    device=device,
    num_classes=5,
    verbose=True,
)

val_metrics = val_eval["metrics"]

# Save metrics JSON
val_json_path = os.path.join(METRICS_DIR, "efficientnet_b0_cropped384_val_metrics.json")
with open(val_json_path, "w") as f:
    json.dump(val_metrics, f, indent=4)
print(f"\nSaved validation metrics JSON: {val_json_path}")

# Plot Confusion Matrix Heatmap
cm = np.array(val_metrics["confusion_matrix"])
fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
ax.figure.colorbar(im, ax=ax)
ax.set(
    xticks=np.arange(len(DR_CLASS_NAMES)),
    yticks=np.arange(len(DR_CLASS_NAMES)),
    xticklabels=DR_CLASS_NAMES,
    yticklabels=DR_CLASS_NAMES,
    title='Validation Confusion Matrix: Cropped 384×384 EfficientNet-B0',
    ylabel='True Label',
    xlabel='Predicted Label'
)
plt.setp(ax.get_xticklabels(), rotation=30, ha="right", rotation_mode="anchor")
thresh = cm.max() / 2.0
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        pct = (cm[i, j] / cm[i].sum() * 100) if cm[i].sum() > 0 else 0
        ax.text(j, i, f"{cm[i, j]}\n({pct:.1f}%)", ha="center", va="center",
                color="white" if cm[i, j] > thresh else "black", fontsize=9)
plt.tight_layout()
cm_path = os.path.join(PLOTS_DIR, "efficientnet_b0_cropped384_confusion_matrix.png")
plt.savefig(cm_path, dpi=300)
plt.show()

## 7. Comparative Metrics Analysis (Baseline vs. Cropped 384×384)

In [ ]:
baseline_json_path = os.path.join(METRICS_DIR, "baseline_val_metrics.json")
if os.path.exists(baseline_json_path):
    with open(baseline_json_path, "r") as f:
        base_m = json.load(f)
else:
    base_m = None

print("=" * 80)
print("          COMPARATIVE VALIDATION PERFORMANCE: BASELINE VS. CROPPED 384x384       ")
print("=" * 80)
headers = ["Metric", "Baseline (224x224 Frozen)", "Cropped 384x384 Fine-Tuned", "Delta"]
rows = []

def format_delta(new, old):
    diff = new - old
    sign = "+" if diff >= 0 else ""
    return f"{sign}{diff:.4f}"

if base_m is not None:
    b_acc = base_m["accuracy"]
    n_acc = val_metrics["accuracy"]
    rows.append(["Accuracy", f"{b_acc:.4f}", f"{n_acc:.4f}", format_delta(n_acc, b_acc)])

    b_f1 = base_m["f1_macro"]
    n_f1 = val_metrics["f1_macro"]
    rows.append(["Macro F1", f"{b_f1:.4f}", f"{n_f1:.4f}", format_delta(n_f1, b_f1)])

    b_qwk = base_m["quadratic_weighted_kappa"]
    n_qwk = val_metrics["quadratic_weighted_kappa"]
    rows.append(["QWK", f"{b_qwk:.4f}", f"{n_qwk:.4f}", format_delta(n_qwk, b_qwk)])

    for c in range(5):
        b_s = base_m["sensitivity_per_class"][c]
        n_s = val_metrics["sensitivity_per_class"][c]
        rows.append([f"Sens (Class {c}: {DR_CLASS_NAMES[c].split(' - ')[-1]})", f"{b_s:.4f}", f"{n_s:.4f}", format_delta(n_s, b_s)])

    df_comp = pd.DataFrame(rows, columns=headers)
    print(df_comp.to_string(index=False))
print("=" * 80)